# 03 · CascadeFlow vs Router Comparison

This notebook compares **CascadeFlow-based routing** vs **your learned router** on the same test set.

It assumes you already have:

- A **router results file** (per-sample decisions, cost, accuracy)
- A **CascadeFlow results file** (from `cascade_flow_experiment.ipynb`),

Both must share a common `sample_id` column so we can align rows.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import json

# 1. File paths (EDIT THESE FOR YOUR ENVIRONMENT)
ROUTER_RESULTS_PATH = Path('outputs/router/router_results.parquet')
CASCADE_RESULTS_PATH = Path('outputs/cascadeflow/cascadeflow_results.parquet')

ROUTER_RESULTS_PATH, CASCADE_RESULTS_PATH


(PosixPath('outputs/router/router_results.parquet'),
 PosixPath('outputs/cascadeflow/cascadeflow_results.parquet'))

In [2]:
# 2. Load router and CascadeFlow results
assert ROUTER_RESULTS_PATH.exists(), f'Router results not found: {ROUTER_RESULTS_PATH}'
assert CASCADE_RESULTS_PATH.exists(), f'CascadeFlow results not found: {CASCADE_RESULTS_PATH}'

router_df = pd.read_parquet(ROUTER_RESULTS_PATH)
cascade_df = pd.read_parquet(CASCADE_RESULTS_PATH)

print('Router df shape:', router_df.shape)
print('Cascade df shape:', cascade_df.shape)

print('\nRouter columns:\n', router_df.columns.tolist())
print('\nCascadeFlow columns:\n', cascade_df.columns.tolist())


AssertionError: Router results not found: outputs/router/router_results.parquet

## 3 · Standardize schema

We want both DataFrames to expose a **common schema** so comparison is easy.

We assume these logical fields:

- `sample_id` – unique sample identifier
- `router_task` – task/category (e.g., `diagram_reasoning`, `ocr`)
- `source_config` – dataset config (e.g., `ai2d`, `textvqa`)

Router-specific columns:
- `router_model` – model chosen by router
- `router_cost` – cost for that sample
- `router_latency_ms` – latency in milliseconds
- `router_is_correct` – boolean correctness

CascadeFlow-specific columns:
- `cascade_model`
- `cascade_cost`
- `cascade_latency_ms`
- `cascade_is_correct`

If your actual column names differ, **edit the mapping dictionaries below**.


In [ ]:
# 3.1 Column mappings (EDIT IF YOUR NAMES DIFFER)

ROUTER_COL_MAP = {
    'sample_id': 'sample_id',
    'router_task': 'router_task',
    'source_config': 'source_config',
    'chosen_model': 'router_model',
    'router_model': 'router_model',
    'total_cost': 'router_cost',
    'cost': 'router_cost',
    'latency_ms': 'router_latency_ms',
    'router_latency_ms': 'router_latency_ms',
    'is_correct': 'router_is_correct',
    'score_f1': 'router_score_f1',
    'glider_score': 'router_glider_score',
}

CASCADE_COL_MAP = {
    'sample_id': 'sample_id',
    'router_task': 'router_task',
    'source_config': 'source_config',
    'cascade_model': 'cascade_model',
    'model_used': 'cascade_model',
    'total_cost': 'cascade_cost',
    'cost': 'cascade_cost',
    'latency_ms': 'cascade_latency_ms',
    'cascade_latency_ms': 'cascade_latency_ms',
    'is_correct': 'cascade_is_correct',
    'score_f1': 'cascade_score_f1',
    'glider_score': 'cascade_glider_score',
}

def remap_columns(df: pd.DataFrame, col_map: dict, prefix: str) -> pd.DataFrame:
    rename_dict = {}
    for old, new in col_map.items():
        if old in df.columns:
            rename_dict[old] = new
    df = df.rename(columns=rename_dict)
    print(f'[{prefix}] Renamed columns:', rename_dict)
    return df

router_std = remap_columns(router_df.copy(), ROUTER_COL_MAP, 'router')
cascade_std = remap_columns(cascade_df.copy(), CASCADE_COL_MAP, 'cascade')

router_std.head(3)


In [ ]:
# 3.2 Check required columns exist
required_router_cols = [
    'sample_id', 'router_task', 'source_config',
    'router_model', 'router_cost', 'router_latency_ms', 'router_is_correct',
]

required_cascade_cols = [
    'sample_id', 'router_task', 'source_config',
    'cascade_model', 'cascade_cost', 'cascade_latency_ms', 'cascade_is_correct',
]

missing_router = [c for c in required_router_cols if c not in router_std.columns]
missing_cascade = [c for c in required_cascade_cols if c not in cascade_std.columns]

print('Missing router columns:', missing_router)
print('Missing cascade columns:', missing_cascade)

assert not missing_router, 'Edit ROUTER_COL_MAP so all required router columns exist.'
assert not missing_cascade, 'Edit CASCADE_COL_MAP so all required cascade columns exist.'


## 4 · Merge router and CascadeFlow results

We align both results on `sample_id` (inner join) so every row has router
and CascadeFlow decisions and metrics.


In [ ]:
# 4. Merge on sample_id
merge_keys = ['sample_id']

merged = router_std.merge(
    cascade_std,
    on=merge_keys,
    how='inner',
    suffixes=('_router', '_cascade'),
)

print('Merged shape:', merged.shape)
merged[['sample_id', 'router_task', 'source_config', 'router_model', 'cascade_model']].head(5)


In [ ]:
# 4.1 Helper columns
merged['match_model_choice'] = merged['router_model'] == merged['cascade_model']

merged['router_is_correct'] = merged['router_is_correct'].astype(bool)
merged['cascade_is_correct'] = merged['cascade_is_correct'].astype(bool)

merged.head(3)


## 5 · Overall comparison: accuracy, cost, latency


In [ ]:
def summarize_overall(df: pd.DataFrame) -> dict:
    n = len(df)
    return {
        'n_samples': int(n),
        'router_accuracy': float(df['router_is_correct'].mean()),
        'cascade_accuracy': float(df['cascade_is_correct'].mean()),
        'router_avg_cost': float(df['router_cost'].mean()),
        'cascade_avg_cost': float(df['cascade_cost'].mean()),
        'router_total_cost': float(df['router_cost'].sum()),
        'cascade_total_cost': float(df['cascade_cost'].sum()),
        'router_avg_latency_ms': float(df['router_latency_ms'].mean()),
        'cascade_avg_latency_ms': float(df['cascade_latency_ms'].mean()),
        'model_choice_agreement': float(df['match_model_choice'].mean()),
    }

overall_summary = summarize_overall(merged)
print(json.dumps(overall_summary, indent=2))


## 6 · Per-task comparison

Group by `router_task` to see where CascadeFlow helps or hurts compared to your router.


In [ ]:
group_cols = ['router_task']

per_task = (
    merged.groupby(group_cols)
    .agg(
        n_samples=('sample_id', 'count'),
        router_accuracy=('router_is_correct', 'mean'),
        cascade_accuracy=('cascade_is_correct', 'mean'),
        router_avg_cost=('router_cost', 'mean'),
        cascade_avg_cost=('cascade_cost', 'mean'),
        router_avg_latency_ms=('router_latency_ms', 'mean'),
        cascade_avg_latency_ms=('cascade_latency_ms', 'mean'),
        model_choice_agreement=('match_model_choice', 'mean'),
    )
    .reset_index()
)

per_task


### 6.1 · Accuracy vs cost per task


In [ ]:
plt.figure()
plt.scatter(per_task['router_avg_cost'], per_task['router_accuracy'], label='Router')
plt.scatter(per_task['cascade_avg_cost'], per_task['cascade_accuracy'], label='CascadeFlow')
for _, row in per_task.iterrows():
    plt.text(row['router_avg_cost'], row['router_accuracy'], row['router_task'], fontsize=8)

plt.xlabel('Average cost per sample')
plt.ylabel('Accuracy')
plt.title('Accuracy vs Cost per Task: Router vs CascadeFlow')
plt.legend()
plt.grid(True)
plt.show()


### 6.2 · Cost and accuracy differences per task


In [ ]:
per_task['delta_accuracy'] = per_task['cascade_accuracy'] - per_task['router_accuracy']
per_task['delta_cost'] = per_task['cascade_avg_cost'] - per_task['router_avg_cost']

per_task.sort_values('delta_accuracy', ascending=False)


In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(per_task['router_task'], per_task['delta_accuracy'])
plt.axhline(0, linestyle='--')
plt.xticks(rotation=45, ha='right')
plt.ylabel('CascadeFlow accuracy - Router accuracy')
plt.title('Accuracy difference per task (positive = CascadeFlow better)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(per_task['router_task'], per_task['delta_cost'])
plt.axhline(0, linestyle='--')
plt.xticks(rotation=45, ha='right')
plt.ylabel('CascadeFlow cost - Router cost')
plt.title('Cost difference per task (negative = CascadeFlow cheaper)')
plt.tight_layout()
plt.show()


## 7 · Model agreement analysis

How often does CascadeFlow pick the **same model** as your router?


In [ ]:
agree = merged[merged['match_model_choice']]
disagree = merged[~merged['match_model_choice']]

def summarize_subset(df: pd.DataFrame, name: str) -> dict:
    if len(df) == 0:
        return {'subset': name, 'n': 0}
    return {
        'subset': name,
        'n': int(len(df)),
        'router_acc': float(df['router_is_correct'].mean()),
        'cascade_acc': float(df['cascade_is_correct'].mean()),
        'router_cost': float(df['router_cost'].mean()),
        'cascade_cost': float(df['cascade_cost'].mean()),
    }

agreement_summary = [
    summarize_subset(agree, 'agree'),
    summarize_subset(disagree, 'disagree'),
]

pd.DataFrame(agreement_summary)


## 8 · Save summary JSON

Save the overall and per-task comparison so it can be used in your paper / report.


In [ ]:
output_dir = Path('outputs/comparison')
output_dir.mkdir(parents=True, exist_ok=True)

summary_payload = {
    'overall': overall_summary,
    'per_task': per_task.to_dict(orient='records'),
    'n_merged': int(len(merged)),
}

summary_path = output_dir / 'router_vs_cascadeflow_summary.json'
with summary_path.open('w') as f:
    json.dump(summary_payload, f, indent=2)

summary_path
